In [8]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, precision_score, recall_score, f1_score

In [9]:
print("Loading testing vectors...")
x_te = joblib.load('../vectors/x_testing_vector.pkl')
y_te = joblib.load('../vectors/y_testing_vector.pkl')
print("Loaded testing vectors")

Loading testing vectors...
Loaded testing vectors


In [10]:
print("Loading saved model and selector...")
sel = joblib.load('../models/sel.pkl')
ensemble = joblib.load('../models/ensemble_model.pkl')
print("Loaded saved model and selector")

Loading saved model and selector...
Loaded saved model and selector


In [11]:
print("Loading individual base models...")
model_names = [
    'Logistic Regression', 'SVM Poly', 'SVM RBF', 'KNN', 'Random Forest', 'Ridge Classifier'
]

best_models = dict()

for name in model_names:
    best_models[name] = joblib.load(f'../models/model_{name}.pkl')

Loading individual base models...


In [12]:
print("Applying feature selection to test data...")
x_te_s = sel.transform(x_te)

Applying feature selection to test data...


In [13]:
def evaluate_model(name, model, x_eval, y_eval):
    y_pred_local = model.predict(x_eval)
    y_prob_local = model.predict_proba(x_eval)

    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_eval, y_pred_local), 4),
        'Precision': round(precision_score(y_eval, y_pred_local), 4),
        'Recall': round(recall_score(y_eval, y_pred_local), 4),
        'F1': round(f1_score(y_eval, y_pred_local), 4),
        'Loss (Log Loss)': round(log_loss(y_eval, y_prob_local), 4),
    }

In [14]:
evaluation_rows = []
for model_name, model in best_models.items():
    evaluation_rows.append(evaluate_model(model_name, model, x_te_s, y_te))
evaluation_rows.append(evaluate_model('Voting Ensemble', ensemble, x_te_s, y_te))

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(by='F1', ascending=False)
print('Evaluation Matrix:')
print(evaluation_df.to_string(index=False))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.5s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Do

Evaluation Matrix:
              Model  Accuracy  Precision  Recall     F1  Loss (Log Loss)
      Random Forest    0.9898     0.9851  0.9957 0.9904           0.0530
    Voting Ensemble    0.9849     0.9801  0.9914 0.9857           0.0550
                KNN    0.9836     0.9743  0.9950 0.9846           0.3421
           SVM Poly    0.9825     0.9733  0.9940 0.9835           0.0556
            SVM RBF    0.9768     0.9715  0.9847 0.9781           0.0710
Logistic Regression    0.9607     0.9597  0.9657 0.9627           0.1004
   Ridge Classifier    0.9551     0.9538  0.9610 0.9574           0.1177


In [15]:
y_pred = ensemble.predict(x_te_s)
y_prob = ensemble.predict_proba(x_te_s)

print('Accuracy:', round(accuracy_score(y_te, y_pred) * 100, 2), '%')
print('Log Loss:', round(log_loss(y_te, y_prob), 4))
print('\nClassification Report:\n', classification_report(y_te, y_pred, target_names=['Ham', 'Spam']))
print('Confusion Matrix:\n', confusion_matrix(y_te, y_pred))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    1.4s finished


Accuracy: 98.49 %
Log Loss: 0.055

Classification Report:
               precision    recall  f1-score   support

         Ham       0.99      0.98      0.98     48476
        Spam       0.98      0.99      0.99     53635

    accuracy                           0.98    102111
   macro avg       0.99      0.98      0.98    102111
weighted avg       0.98      0.98      0.98    102111

Confusion Matrix:
 [[47397  1079]
 [  463 53172]]


In [16]:
evaluation_df.to_csv('../reports/model_report.csv', index=False)
print("Saved leaderboard to 'model_report.csv'")

with open('../reports/ensemble_report.txt', 'w') as f:
    f.write("--- Accuracy & Log Loss ---\n")
    f.write(f"Accuracy: {round(accuracy_score(y_te, y_pred) * 100, 2)}%\n")
    f.write(f"Log Loss: {round(log_loss(y_te, y_prob), 4)}\n\n")
    
    f.write("--- Classification Report ---\n")
    f.write(classification_report(y_te, y_pred, target_names=['Ham', 'Spam']))
    f.write("\n\n")
    
    f.write("--- Confusion Matrix ---\n")
    cm = confusion_matrix(y_te, y_pred)
    f.write(f"True Ham  (Correctly identified ham emails): {cm[0][0]}\n")
    f.write(f"False Spam (Incorrectly identified hams as spams): {cm[0][1]}\n")
    f.write(f"False Ham  (Incorrectly identified spams as hams): {cm[1][0]}\n")
    f.write(f"True Spam (Correctly identified spam emails): {cm[1][1]}\n")

print("Saved detailed ensemble statistics to 'ensemble_report.txt'")

Saved leaderboard to 'model_report.csv'
Saved detailed ensemble statistics to 'ensemble_report.txt'


In [9]:
!jupyter nbconvert --to script model-testing.ipynb

[NbConvertApp] Converting notebook model-testing.ipynb to script
[NbConvertApp] Writing 2183 bytes to model-testing.py
